# ML + QAOA Drug Candidate Selection Using IBM Qiskit

This notebook demonstrates a simple pharma-inspired workflow:

1. Use **classical machine learning** to predict molecular solubility.
2. Use **classical optimization baselines** to solve the candidate-selection problem.
3. Use **QAOA** to select the best small portfolio of molecules from the ML-scored candidates.

That distinction matters: QAOA does **not** predict solubility. QAOA solves the selection optimization problem after we already have candidate scores.

Dataset: Delaney ESOL from MoleculeNet/DeepChem.

CSV source: https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv

Documentation: https://deepchem.readthedocs.io/en/latest/api_reference/moleculenet.html



## Workflow

The project is split into two problems:

| Step | Question | Method |
|---|---|---|
| Prediction | What is a molecule's likely measured log solubility? | Classical ML regression |
| Optimization baseline | What is the exact best three-molecule selection? | Classical brute force enumeration |
| Optimization heuristic | Can a scalable classical heuristic find the same selection? | Simulated annealing |
| Quantum optimization | Can QAOA recover a high-scoring feasible selection? | QAOA |

We compare ML models for prediction quality, then compare simulated annealing and QAOA against the exact classical optimizer for selection quality.



In [ ]:
from pathlib import Path
import itertools
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load the ESOL Dataset

ESOL contains real molecules, SMILES strings, measured aqueous solubility, and simple molecular descriptors. The local CSV is used if present; otherwise the notebook falls back to the public URL.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
local_paths = [Path("data/delaney-processed.csv"), Path("../data/delaney-processed.csv")]
data_path = next((path for path in local_paths if path.exists()), None)

df = pd.read_csv(data_path if data_path else DATA_URL)

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

## 2. Train Classical ML Baselines

We train a few simple regression models to predict measured log solubility. These are the prediction baselines. QAOA is not involved in this step.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_columns = [
    "ESOL predicted log solubility in mols per litre",
    "Minimum Degree",
    "Molecular Weight",
    "Number of H-Bond Donors",
    "Number of Rings",
    "Number of Rotatable Bonds",
    "Polar Surface Area",
]
target_column = "measured log solubility in mols per litre"

X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED
)

models = {
    "Linear Regression": Pipeline(
        [("scaler", StandardScaler()), ("model", LinearRegression())]
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=250, random_state=RANDOM_SEED, min_samples_leaf=2
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_SEED),
}

metrics = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    metrics.append(
        {
            "model": name,
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": rmse,
            "R2": r2_score(y_test, predictions),
        }
    )
    trained_models[name] = model

metrics_df = pd.DataFrame(metrics).sort_values("RMSE").reset_index(drop=True)
best_model_name = metrics_df.loc[0, "model"]
best_model = trained_models[best_model_name]

print("Best model by test RMSE:", best_model_name)
metrics_df

## 3. Inspect Prediction Quality

The plot compares measured solubility against predictions from the best classical ML model.


In [ ]:
best_predictions = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_predictions, alpha=0.7, edgecolor="white", linewidth=0.5)
limits = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
ax.plot(limits, limits, color="black", linewidth=1)
ax.set_title(f"Measured vs Predicted Solubility ({best_model_name})")
ax.set_xlabel("Measured log solubility")
ax.set_ylabel("Predicted log solubility")
plt.tight_layout()
plt.show()

## 4. Build an Out-of-Sample Candidate Pool

Now we move from prediction to optimization. We take candidate molecules from the test set so their ML scores are out-of-sample predictions.


In [ ]:
test_frame = df.loc[X_test.index].copy()
test_frame["predicted_log_solubility"] = best_predictions

filtered_test = test_frame[
    test_frame["Molecular Weight"].between(150, 500)
    & (test_frame["Number of H-Bond Donors"] <= 5)
    & (test_frame["Number of Rotatable Bonds"] <= 8)
    & (test_frame["Number of Rings"] >= 1)
].copy()

candidate_pool = filtered_test.sample(8, random_state=RANDOM_SEED).reset_index(drop=True)
candidate_pool["variable"] = [f"x_{i}" for i in candidate_pool.index]

candidate_pool[[
    "variable",
    "Compound ID",
    "smiles",
    "measured log solubility in mols per litre",
    "predicted_log_solubility",
    "Molecular Weight",
    "Polar Surface Area",
    "Number of Rotatable Bonds",
]]

## 5. Create Selection Scores

The optimizer needs one score per molecule. We combine the ML-predicted solubility with simple developability preferences.

This is intentionally simple and interpretable:

- higher predicted solubility is better
- molecular weight closer to 350 Da is preferred
- polar surface area closer to 75 is preferred
- fewer rotatable bonds are preferred


In [ ]:
def minmax(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(np.ones(len(series)), index=series.index)
    return (series - series.min()) / span


candidate_pool["predicted_solubility_score"] = minmax(
    candidate_pool["predicted_log_solubility"]
)
candidate_pool["mw_desirability"] = 1 - minmax(
    (candidate_pool["Molecular Weight"] - 350).abs()
)
candidate_pool["psa_desirability"] = 1 - minmax(
    (candidate_pool["Polar Surface Area"] - 75).abs()
)
candidate_pool["rotatable_desirability"] = 1 - minmax(
    candidate_pool["Number of Rotatable Bonds"]
)

candidate_pool["selection_score"] = (
    5 * candidate_pool["predicted_solubility_score"]
    + 2 * candidate_pool["mw_desirability"]
    + 2 * candidate_pool["psa_desirability"]
    + candidate_pool["rotatable_desirability"]
)

candidate_pool[[
    "variable",
    "Compound ID",
    "predicted_log_solubility",
    "measured log solubility in mols per litre",
    "Molecular Weight",
    "Polar Surface Area",
    "Number of Rotatable Bonds",
    "selection_score",
]].sort_values("selection_score", ascending=False)

## 6. Classical Optimizer Baseline

This is the right baseline for QAOA. Since we only have eight candidates, brute force can enumerate all possible groups of three and find the exact best selection.


In [ ]:
def score_selection(indices, frame):
    return float(frame.loc[list(indices), "selection_score"].sum())


all_solutions = []
for combo in itertools.combinations(candidate_pool.index, 3):
    selected_names = candidate_pool.loc[list(combo), "Compound ID"].tolist()
    all_solutions.append(
        {
            "selected_indices": combo,
            "selected_candidates": selected_names,
            "score": score_selection(combo, candidate_pool),
        }
    )

classical_selection = (
    pd.DataFrame(all_solutions).sort_values("score", ascending=False).reset_index(drop=True)
)
best_classical = classical_selection.iloc[0]

print("Classical optimizer optimum")
print("Selected:", best_classical["selected_candidates"])
print("Score:", round(best_classical["score"], 4))
classical_selection.head(10)

## 7. Simulated Annealing Baseline

Brute force is exact, but it only works because this demo has eight candidates. Simulated annealing is a more scalable classical heuristic: it starts with a valid three-molecule portfolio, proposes swaps, and sometimes accepts a worse swap early in the search so it can escape local optima.

Here the neighborhood is simple: remove one selected molecule and add one unselected molecule. That keeps the `select exactly three` constraint satisfied at every step.



In [ ]:
def simulated_annealing_selection(
    frame,
    select_count=3,
    iterations=2_000,
    initial_temperature=1.0,
    cooling_rate=0.995,
    min_temperature=1e-4,
    seed=RANDOM_SEED,
):
    rng = np.random.default_rng(seed)
    values = frame["selection_score"].to_numpy(dtype=float)
    n_candidates = len(values)

    current_bits = np.zeros(n_candidates, dtype=int)
    current_bits[rng.choice(n_candidates, size=select_count, replace=False)] = 1

    def bitstring_score(bits):
        return float(values[bits == 1].sum())

    current_score = bitstring_score(current_bits)
    best_bits = current_bits.copy()
    best_score = current_score
    temperature = initial_temperature
    trace = []

    for step in range(1, iterations + 1):
        proposal_bits = current_bits.copy()
        selected = np.flatnonzero(proposal_bits == 1)
        unselected = np.flatnonzero(proposal_bits == 0)

        remove_idx = rng.choice(selected)
        add_idx = rng.choice(unselected)
        proposal_bits[remove_idx] = 0
        proposal_bits[add_idx] = 1

        proposal_score = bitstring_score(proposal_bits)
        score_delta = proposal_score - current_score
        accept_worse_probability = (
            1.0
            if score_delta >= 0
            else np.exp(score_delta / max(temperature, min_temperature))
        )

        if rng.random() < accept_worse_probability:
            current_bits = proposal_bits
            current_score = proposal_score

            if current_score > best_score:
                best_bits = current_bits.copy()
                best_score = current_score

        trace.append(
            {
                "step": step,
                "temperature": temperature,
                "current_score": current_score,
                "best_score": best_score,
            }
        )
        temperature = max(temperature * cooling_rate, min_temperature)

    return best_bits, best_score, pd.DataFrame(trace)


annealing_bits, annealing_score, annealing_trace = simulated_annealing_selection(
    candidate_pool
)
annealing_selected = candidate_pool.loc[annealing_bits == 1].copy()

print("Simulated annealing selected candidates:")
display(
    annealing_selected[[
        "Compound ID",
        "smiles",
        "predicted_log_solubility",
        "measured log solubility in mols per litre",
        "selection_score",
    ]]
)
print("Simulated annealing score:", round(annealing_score, 4))
print("Classical optimizer best score:", round(best_classical["score"], 4))
print("Matches classical optimum:", np.isclose(annealing_score, best_classical["score"]))

annealing_trace.tail()



## 8. Build the Qiskit Optimization Problem

We express the selection problem with binary variables and the constraint that exactly three molecules must be selected.


In [ ]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

problem = QuadraticProgram("ml_scored_esol_candidate_selection")

for variable in candidate_pool["variable"]:
    problem.binary_var(name=variable)

linear_objective = {
    row.variable: float(row.selection_score)
    for row in candidate_pool.itertuples(index=False)
}

problem.maximize(linear=linear_objective)
problem.linear_constraint(
    linear={variable: 1 for variable in candidate_pool["variable"]},
    sense="==",
    rhs=3,
    name="select_exactly_three",
)

print(problem.prettyprint())

In [ ]:
qubo_converter = QuadraticProgramToQubo()
qubo = qubo_converter.convert(problem)
print(qubo.prettyprint())

## 9. Solve the Selection Problem with QAOA

QAOA now receives the ML-derived selection problem. It is optimizing a subset choice, not predicting solubility.


In [ ]:
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer


def make_sampler(seed=RANDOM_SEED):
    """Create a sampler across common Qiskit versions."""
    try:
        from qiskit.primitives import Sampler

        return Sampler(options={"seed": seed, "shots": 4096})
    except Exception:
        from qiskit.primitives import StatevectorSampler

        return StatevectorSampler(seed=seed)


sampler = make_sampler()
optimizer = COBYLA(maxiter=200)
qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=3)

qaoa_optimizer = MinimumEigenOptimizer(qaoa)
qaoa_result = qaoa_optimizer.solve(problem)

print(qaoa_result)

## 10. Compare Optimization Results


In [ ]:
selection_bits = np.array(qaoa_result.x).astype(int)
qaoa_selected = candidate_pool.loc[selection_bits == 1].copy()
qaoa_score = float(qaoa_selected["selection_score"].sum())

comparison = pd.DataFrame(
    [
        {
            "method": "Exact brute force",
            "score": float(best_classical["score"]),
            "selected_candidates": best_classical["selected_candidates"],
            "matches_exact": True,
        },
        {
            "method": "Simulated annealing",
            "score": annealing_score,
            "selected_candidates": annealing_selected["Compound ID"].tolist(),
            "matches_exact": np.isclose(annealing_score, best_classical["score"]),
        },
        {
            "method": "QAOA",
            "score": qaoa_score,
            "selected_candidates": qaoa_selected["Compound ID"].tolist(),
            "matches_exact": np.isclose(qaoa_score, best_classical["score"]),
        },
    ]
)
comparison["optimality_gap"] = float(best_classical["score"]) - comparison["score"]
comparison["optimality_gap_pct"] = (
    100 * comparison["optimality_gap"] / float(best_classical["score"])
)

display(comparison)

print("QAOA selected candidates:")
display(
    qaoa_selected[[
        "Compound ID",
        "smiles",
        "predicted_log_solubility",
        "measured log solubility in mols per litre",
        "selection_score",
    ]]
)


## 11. Visualize Candidate Selection

Candidates selected by QAOA are highlighted in green. Candidates selected by simulated annealing are marked with an outline.


In [ ]:
plot_frame = candidate_pool.copy()
plot_frame["selected_by_qaoa"] = selection_bits == 1
plot_frame["selected_by_annealing"] = annealing_bits == 1
plot_frame = plot_frame.sort_values("selection_score", ascending=True)

colors = np.where(plot_frame["selected_by_qaoa"], "#2e7d32", "#9e9e9e")
edge_colors = np.where(plot_frame["selected_by_annealing"], "#1565c0", "#616161")
line_widths = np.where(plot_frame["selected_by_annealing"], 2.5, 0.8)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(
    plot_frame["Compound ID"],
    plot_frame["selection_score"],
    color=colors,
    edgecolor=edge_colors,
    linewidth=line_widths,
)
ax.set_title("ML-Scored ESOL Candidate Selection")
ax.set_xlabel("Selection score")
ax.set_ylabel("Compound")

for bar, value in zip(bars, plot_frame["selection_score"]):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        va="center",
    )

legend_handles = [
    plt.Line2D([0], [0], color="#2e7d32", lw=8, label="QAOA selected"),
    plt.Line2D([0], [0], color="#1565c0", lw=2.5, label="Simulated annealing outline"),
]
ax.legend(handles=legend_handles, loc="lower right")

plt.tight_layout()
plt.show()


## 12. Conclusion

This notebook uses classical ML for prediction, classical baselines for optimization comparison, and QAOA for quantum optimization.

- Classical ML predicts measured log solubility from molecular descriptors.
- The ML prediction becomes part of a selection score.
- A classical brute-force optimizer gives the exact best selection for this tiny candidate pool.
- Simulated annealing provides a scalable classical heuristic baseline.
- QAOA attempts to recover the same selection as a quantum optimization approach.
- Because QAOA is approximate and shallow here, it may find a strong feasible selection without always matching the exact classical optimum.

In a larger pharma workflow, the prediction model could use richer molecular fingerprints, docking scores, ADMET predictions, or graph neural network embeddings. The QAOA problem could include constraints such as budget, novelty, toxicity limits, or portfolio diversity.
